In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Load the correlated results we saved
corr_df = pd.read_csv('../data/flagged_trades_correlated.csv')

print(f"Loaded {len(corr_df)} flagged trading records")
print(corr_df.dtypes)
print(corr_df.head())

Loaded 78 flagged trading records
date                  str
bank                  str
ticker                str
price_move_pct    float64
volume_ratio      float64
event_type            str
event_detail          str
days_to_event     float64
dtype: object
         date       bank ticker  price_move_pct  volume_ratio  \
0  2023-04-14  JP Morgan    JPM            7.55          2.62   
1  2023-08-15  JP Morgan    JPM           -2.55          1.20   
2  2023-10-13  JP Morgan    JPM            1.50          3.08   
3  2023-10-27  JP Morgan    JPM           -3.60          1.55   
4  2024-01-12  JP Morgan    JPM           -0.73          1.89   

              event_type      event_detail  days_to_event  
0  Earnings Announcement  Q1 2023 Earnings            0.0  
1     No event in window               NaN            NaN  
2  Earnings Announcement  Q3 2023 Earnings            0.0  
3     No event in window               NaN            NaN  
4  Earnings Announcement  Q4 2023 Earnings           

In [3]:
# ── Assign risk levels based on business rules ───────────────
# This mimics how a real compliance team would tier alerts
# High   = large move + high volume + near earnings (most suspicious)
# Medium = moderate anomaly + some event correlation
# Low    = flagged by model but weak signals

def assign_risk(row):
    score = 0
    
    # Volume signal — biggest insider trading indicator
    if row['volume_ratio'] >= 3.0:
        score += 3
    elif row['volume_ratio'] >= 2.0:
        score += 2
    else:
        score += 1
    
    # Price move signal
    if abs(row['price_move_pct']) >= 5.0:
        score += 3
    elif abs(row['price_move_pct']) >= 2.0:
        score += 2
    else:
        score += 1
    
    # Corporate event proximity — key FCA MAR signal
    if row['event_type'] == 'Earnings Announcement':
        if isinstance(row['days_to_event'], (int, float)) and row['days_to_event'] <= 2:
            score += 4  # Trading 1-2 days before earnings = very suspicious
        else:
            score += 2
    elif row['event_type'] == 'Dividend Payment':
        score += 1
    
    # Convert score to risk tier
    if score >= 7:
        return 'HIGH'
    elif score >= 4:
        return 'MEDIUM'
    else:
        return 'LOW'

corr_df['risk_level'] = corr_df.apply(assign_risk, axis=1)

print("Risk level distribution:")
print(corr_df['risk_level'].value_counts())
print(f"\nSample HIGH risk flags:")
print(corr_df[corr_df['risk_level'] == 'HIGH'][
    ['date','bank','price_move_pct','volume_ratio','event_type','days_to_event']
].head(8))

Risk level distribution:
risk_level
LOW       38
MEDIUM    22
HIGH      18
Name: count, dtype: int64

Sample HIGH risk flags:
          date           bank  price_move_pct  volume_ratio  \
0   2023-04-14      JP Morgan            7.55          2.62   
2   2023-10-13      JP Morgan            1.50          3.08   
5   2024-04-12      JP Morgan           -6.47          3.16   
8   2024-10-11      JP Morgan            4.44          2.02   
14  2023-04-18  Goldman Sachs           -1.70          3.38   
19  2024-04-15  Goldman Sachs            2.92          2.74   
28  2023-02-15       Barclays           -7.86          3.14   
38  2023-10-24       Barclays           -6.53          2.36   

               event_type  days_to_event  
0   Earnings Announcement            0.0  
2   Earnings Announcement            0.0  
5   Earnings Announcement            0.0  
8   Earnings Announcement            0.0  
14  Earnings Announcement            0.0  
19  Earnings Announcement            0.0  
28  E

In [4]:
# ── Encode categorical features ───────────────────────────────
df = corr_df.copy()

# Encode bank names
bank_encoder = LabelEncoder()
df['bank_encoded'] = bank_encoder.fit_transform(df['bank'])

# Encode event type
event_encoder = LabelEncoder()
df['event_encoded'] = event_encoder.fit_transform(df['event_type'])

# Clean days_to_event — fill N/A with 999 (no event nearby)
df['days_to_event_clean'] = pd.to_numeric(df['days_to_event'], errors='coerce').fillna(999)

# ── Define features and target ────────────────────────────────
feature_cols = [
    'price_move_pct',
    'volume_ratio',
    'bank_encoded',
    'event_encoded',
    'days_to_event_clean'
]

# Encode target
risk_encoder = LabelEncoder()
df['risk_encoded'] = risk_encoder.fit_transform(df['risk_level'])

X = df[feature_cols]
y = df['risk_encoded']

print("Feature matrix shape:", X.shape)
print("Target classes:", risk_encoder.classes_)
print("Class distribution:", dict(zip(risk_encoder.classes_, 
      [sum(y==i) for i in range(len(risk_encoder.classes_))])))
print(X.head())

Feature matrix shape: (78, 5)
Target classes: ['HIGH' 'LOW' 'MEDIUM']
Class distribution: {'HIGH': 18, 'LOW': 38, 'MEDIUM': 22}
   price_move_pct  volume_ratio  bank_encoded  event_encoded  \
0            7.55          2.62             3              0   
1           -2.55          1.20             3              1   
2            1.50          3.08             3              0   
3           -3.60          1.55             3              1   
4           -0.73          1.89             3              0   

   days_to_event_clean  
0                  0.0  
1                999.0  
2                  0.0  
3                999.0  
4                  0.0  


In [5]:
# ── Train/test split ─────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples:  {len(X_test)}")

# ── Train XGBoost ─────────────────────────────────────────────
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42
)

model.fit(X_train, y_train)

# ── Evaluate ──────────────────────────────────────────────────
y_pred = model.predict(X_test)

print("\n=== Classification Report ===")
print(classification_report(
    y_test, y_pred,
    target_names=risk_encoder.classes_
))

# Feature importance
print("\n=== Feature Importance ===")
importance_df = pd.DataFrame({
    'feature':   feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
print(importance_df)

Training samples: 62
Testing samples:  16

=== Classification Report ===
              precision    recall  f1-score   support

        HIGH       1.00      1.00      1.00         4
         LOW       0.89      1.00      0.94         8
      MEDIUM       1.00      0.75      0.86         4

    accuracy                           0.94        16
   macro avg       0.96      0.92      0.93        16
weighted avg       0.94      0.94      0.93        16


=== Feature Importance ===
               feature  importance
3        event_encoded    0.702473
1         volume_ratio    0.152941
0       price_move_pct    0.097732
4  days_to_event_clean    0.031633
2         bank_encoded    0.015221
